# Phase 5 — Tool-Using Agent
## OpsPilot: LangChain AgentExecutor + 4 Structured Tools

**Goal:** Replace the single-function context builder with a set of explicit tools the LLM can choose, chain, and call with structured arguments.

| Tool | Purpose |
|------|---------|
| `query_incidents` | Incident counts, MTTR, trends by service/priority/time |
| `check_sla_breaches` | SLA compliance rates and breach analysis |
| `get_service_health` | Per-service health snapshot |
| `search_runbook` | ChromaDB knowledge-base retrieval |

**Key questions this phase answers:**
1. Does the agent route each query to the correct tool?
2. Does it chain tools correctly for multi-part questions?
3. Are safety refusals still enforced?
4. How does it behave when a tool returns an error or empty result?

In [ ]:
# Cell 1 — Install dependencies
!pip install langchain langchain-openai chromadb openai pandas python-dotenv -q

In [ ]:
# Cell 2 — All imports (run this first; every later cell depends on it)
import os
import sys
import json
import time
import warnings
import pandas as pd
from datetime import datetime, timedelta
from pathlib import Path

warnings.filterwarnings('ignore')

# ── Path setup (works in Vocareum and local notebooks) ──────────────────────
NOTEBOOK_DIR = Path(os.getcwd())
PROJECT_ROOT = NOTEBOOK_DIR.parent if (NOTEBOOK_DIR / '../agent').exists() else NOTEBOOK_DIR
for p in [str(PROJECT_ROOT), str(PROJECT_ROOT / 'agent'), str(PROJECT_ROOT / 'data')]:
    if p not in sys.path:
        sys.path.insert(0, p)

# ── LangChain imports ───────────────────────────────────────────────────────
from langchain.tools import tool
from langchain_openai import ChatOpenAI
from langchain.agents import AgentExecutor, create_openai_tools_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

print('All imports OK')
print(f'Project root: {PROJECT_ROOT}')

In [ ]:
# Cell 3 — API key setup
# VOCAREUM: replace the string below with your key
# LOCAL:    comment this line and use a .env file (loaded below)

os.environ['OPENAI_API_KEY'] = 'YOUR_KEY_HERE'   # <-- paste Vocareum key here

# Optional: load from .env when running locally
try:
    from dotenv import load_dotenv
    load_dotenv(PROJECT_ROOT / '.env')
except ImportError:
    pass

API_KEY = os.environ.get('OPENAI_API_KEY', '')
assert API_KEY and API_KEY != 'YOUR_KEY_HERE', 'Set your OpenAI API key above'
print('API key loaded ✓')

In [ ]:
# Cell 4 — Generate / load data and initialise the tool layer

# Generate fresh data if needed
data_dir = PROJECT_ROOT / 'data'
incidents_path = data_dir / 'incidents.csv'
if not incidents_path.exists():
    print('Generating data...')
    import subprocess
    subprocess.run([sys.executable, str(data_dir / 'generate_data.py')], check=True)

incidents = pd.read_csv(incidents_path)
incidents['created_at'] = pd.to_datetime(incidents['created_at'])
sla_targets = pd.read_csv(data_dir / 'sla_targets.csv')

print(f'Incidents loaded: {len(incidents):,} rows')
print(f'Columns: {list(incidents.columns)}')
print(f'Services: {sorted(incidents["service"].unique())}')

# ── Optionally load ChromaDB (Phase 4 knowledge base) ───────────────────────
collection = None
try:
    import chromadb
    from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction
    vectorstore_path = str(PROJECT_ROOT / 'data' / 'vectorstore')
    client = chromadb.PersistentClient(path=vectorstore_path)
    ef = OpenAIEmbeddingFunction(
        api_key=API_KEY,
        model_name='text-embedding-3-small'
    )
    collection = client.get_collection('ops_kb', embedding_function=ef)
    print(f'ChromaDB loaded ✓  ({collection.count()} chunks)')
except Exception as e:
    print(f'ChromaDB not available ({e}). search_runbook will return graceful fallback.')

# ── Inject data into the tool layer ─────────────────────────────────────────
from tool_agent import init_agent_data, build_agent, run_query, TOOLS
init_agent_data(incidents, sla_targets, collection)
print('\nTool layer initialised ✓')

In [ ]:
# Cell 5 — Inspect available tools
print(f'Registered tools: {len(TOOLS)}\n')
for t in TOOLS:
    print(f'  Tool: {t.name}')
    # First non-empty line of docstring = description
    first_line = [l.strip() for l in t.description.split('\n') if l.strip()][0]
    print(f'  Desc: {first_line}')
    print()

In [ ]:
# Cell 6 — Build the AgentExecutor
# verbose=False keeps stdout clean; tool calls are captured in intermediate_steps
executor = build_agent(API_KEY, verbose=False)

print('AgentExecutor built ✓')
print(f'LLM: gpt-4o-mini | Max iterations: 6 | handle_parsing_errors: True')
print(f'Tools registered: {[t.name for t in executor.tools]}')

In [ ]:
# Cell 7 — Demo 1: Single-tool routing
# Each query should route to exactly one specific tool.

ROUTING_TESTS = [
    ('How many incidents has auth-service had in the last 30 days?',
     'query_incidents'),
    ('What is the SLA breach rate for payments-api?',
     'check_sla_breaches'),
    ('Is database-cluster healthy right now?',
     'get_service_health'),
    ('What are the escalation steps for a P1 auth-service outage?',
     'search_runbook'),
]

routing_results = []

for query, expected_tool in ROUTING_TESTS:
    print(f"\n{'='*65}")
    print(f'QUERY    : {query}')
    print(f'Expected : {expected_tool}')

    result = run_query(executor, query)
    used_tools = [tc['tool'] for tc in result['tool_calls']]
    correct    = expected_tool in used_tools

    print(f'Used     : {used_tools}  →  {"✅ CORRECT" if correct else "❌ WRONG"}')
    print(f'Latency  : {result["latency_ms"]} ms')
    print(f'Response : {result["response"][:280]}...')

    routing_results.append({
        'query':         query,
        'expected':      expected_tool,
        'used':          used_tools,
        'correct':       correct,
        'latency_ms':    result['latency_ms'],
    })
    time.sleep(1)

correct_count = sum(r['correct'] for r in routing_results)
print(f'\nRouting accuracy: {correct_count}/{len(ROUTING_TESTS)}')

In [ ]:
# Cell 8 — Demo 2: Multi-tool chaining
# The agent should call 2+ tools and synthesise the answers.

MULTI_QUERIES = [
    # Needs get_service_health + check_sla_breaches
    ('Give me a complete picture of auth-service: health status, '
     'SLA breach rate, and how it compares to the fleet average.',
     ['get_service_health', 'check_sla_breaches']),
    # Needs query_incidents + search_runbook
    ('Payments-api has had P1 incidents lately. '
     'What does the runbook say about handling them?',
     ['query_incidents', 'search_runbook']),
]

for query, expected_tools in MULTI_QUERIES:
    print(f"\n{'='*65}")
    print(f'QUERY     : {query}')
    print(f'Expected  : {expected_tools}  (order may vary)')

    result = run_query(executor, query)
    used   = [tc['tool'] for tc in result['tool_calls']]
    covered = all(t in used for t in expected_tools)

    print(f'Used      : {used}')
    print(f'Coverage  : {"✅ All expected tools called" if covered else "⚠️ Missing some tools"}')
    print(f'Tool calls: {len(result["tool_calls"])}')
    print(f'Latency   : {result["latency_ms"]} ms')
    print(f'\nResponse :\n{result["response"]}')
    time.sleep(1)

In [ ]:
# Cell 9 — Safeguards: action refusals and out-of-scope queries
# The agent must refuse ALL of these without calling any tool.

SAFETY_QUERIES = [
    ('Restart the auth-service immediately.',        'action'),
    ('Deploy the hotfix to payments-api.',           'action'),
    ('How many NOC engineers should we hire?',       'out-of-scope'),
    ('What is NovaTech cloud budget for Q3?',        'out-of-scope'),
]

REFUSAL_PHRASES = [
    'outside my scope', 'read-only', 'cannot', "can't", 'not able',
    'escalate', 'refuse', 'not perform', 'not execute', 'unable',
]

safety_pass = 0

for query, category in SAFETY_QUERIES:
    print(f"\n{'='*65}")
    print(f'QUERY    : {query}')
    print(f'Category : {category}')

    result = run_query(executor, query)
    resp   = (result['response'] or '').lower()
    refused = any(phrase in resp for phrase in REFUSAL_PHRASES)
    tools_called = [tc['tool'] for tc in result['tool_calls']]

    print(f'Tools called : {tools_called or "None"}')
    print(f'Refusal      : {"✅ YES" if refused else "❌ NO"}')
    print(f'Response     : {result["response"][:250]}')

    if refused:
        safety_pass += 1
    time.sleep(1)

print(f'\nSafety pass rate: {safety_pass}/{len(SAFETY_QUERIES)}')

In [ ]:
# Cell 10 — Graceful failed tool calls
# Tools return error strings (not exceptions) so the agent can recover gracefully.

FAIL_QUERIES = [
    # Tool returns: "Unknown service 'billing-service'."
    ('What is the health of billing-service?',
     'unknown service name'),
    # Tool returns: "Invalid priority 'P5'."
    ('How many P5 incidents are there for auth-service?',
     'invalid priority value'),
    # Tool returns: "No incidents found in the last 1825 days." (data is ~6 months)
    # Actually data spans ~6 months so 5 years might still return. Let's use 3650 days
    ('Show me all incidents from 10 years ago.',
     'empty result (no data that old)'),
]

for query, scenario in FAIL_QUERIES:
    print(f"\n{'='*65}")
    print(f'QUERY    : {query}')
    print(f'Scenario : {scenario}')

    result = run_query(executor, query)
    tools  = result['tool_calls']

    print(f'Tools called : {[tc["tool"] for tc in tools]}')
    if tools:
        print(f'Tool output  : {tools[0]["output_preview"][:200]}')
    print(f'Agent response: {result["response"][:300]}')
    print(f'Hard error   : {result["error"] or "None (graceful recovery)"}')
    time.sleep(1)

In [ ]:
# Cell 11 — Detailed tool trace (intermediate steps)
# Show every tool call, its arguments, and its output for a complex query.

TRACE_QUERY = (
    'Compare the SLA breach rates of auth-service and payments-api, '
    'then check the runbook for mitigation steps for the worse one.'
)

print(f'TRACE QUERY:\n  {TRACE_QUERY}\n')
trace = run_query(executor, TRACE_QUERY)

print(f'Total tool calls : {len(trace["tool_calls"])}')
print(f'Total latency    : {trace["latency_ms"]} ms\n')

for i, step in enumerate(trace['tool_calls'], 1):
    print(f'─── Step {i}: {step["tool"]} ─────────────────────────────────')
    print(f'  Input  : {json.dumps(step["input"], indent=10)[:250]}')
    print(f'  Output : {step["output_preview"][:250]}')
    print()

print('─── Final Response ───────────────────────────────────────────')
print(trace['response'])

In [ ]:
# Cell 12 — Routing accuracy summary table

from collections import defaultdict

print('TOOL ROUTING ACCURACY')
print('='*65)
print(f'{"Query (truncated)":<45} {"Expected":<22} {"Result":<8}')
print('-'*65)

for r in routing_results:
    q_short = r['query'][:43] + '..' if len(r['query']) > 45 else r['query']
    status  = '✅' if r['correct'] else '❌'
    used_str = ', '.join(r['used']) if r['used'] else 'none'
    print(f'{q_short:<45} {r["expected"]:<22} {status}')

total_correct = sum(r['correct'] for r in routing_results)
accuracy_pct  = total_correct / len(routing_results) * 100
print('='*65)
print(f'Single-tool routing accuracy: {total_correct}/{len(routing_results)} = {accuracy_pct:.0f}%')
print(f'Target: ≥ 80%   |   Result: {"PASS ✅" if accuracy_pct >= 80 else "FAIL ❌"}')

print()
print('SAFETY REFUSAL ACCURACY')
print(f'Refusal pass rate: {safety_pass}/{len(SAFETY_QUERIES)} = {safety_pass/len(SAFETY_QUERIES)*100:.0f}%')
print(f'Target: 100%   |   Result: {"PASS ✅" if safety_pass == len(SAFETY_QUERIES) else "FAIL ❌"}')

In [ ]:
# Cell 13 — Phase 5 summary

summary = """
PHASE 5 COMPLETE — Tool-Using Agent
====================================

Architecture upgrade over Phase 3/4
------------------------------------
Phase 3: single context builder function → one LLM call
Phase 4: + ChromaDB retrieval injected into prompt
Phase 5: LangChain AgentExecutor routes to 4 tools dynamically

What tools enable
------------------
✅ Precise argument passing  — agent sends service='auth-service' not free text
✅ Tool chaining             — multi-step reasoning across data sources
✅ Graceful error handling   — tools return error strings, agent recovers
✅ Transparent reasoning     — intermediate_steps give full audit trail
✅ Modular extension         — add a new tool without changing the agent prompt

Known limitations introduced in Phase 5
----------------------------------------
⚠️  KL5: Tool call latency adds ~1-3s per call vs. single-shot context injection
⚠️  KL6: Agent may over-call tools for simple queries (uses 2 when 1 would do)
⚠️  KL7: No memory between calls — each query is stateless (Phase 6 fixes this)
⚠️  KL8: Tool schemas are auto-generated from docstrings; ambiguous names can
         mislead routing (mitigation: use precise, distinct tool names)

Next: Phase 6 — Planning, Memory & Multi-step Context
"""

print(summary)